In [ ]:
import numpy as np
from tqdm import tqdm

from alias import *
from utils import *

In [ ]:
dfs = []
plan = make_plan(
    [
        ("Simple", bal.Simple, est.DifferenceInMeans, {}),
        ("Efron", bal.Efron, est.DifferenceInMeans, {}),
        ("QuickBlock", bal.QuickBlock, est.Blocking, {}),
        ("Alweiss", bal.Alweiss, est.DifferenceInMeans, {}),
        ("BWD", bal.BWD, est.DifferenceInMeans, {}),
    ]
)

dgp_factory_class_list = [dgp.LinearFactory]

NUM_ITERS = 5

# sample_sizes = [50, 100, 250, 500, 1000, 2_500, 5_000, 10_000, 25_000, 50_000, 100_000, 500_000, 1_000_000, 5_000_000]
sample_sizes = np.concatenate(
    (
        np.round(
            np.logspace(np.log10(50), np.log10(1_000_000_000), base=10, num=15)
        ).astype(int),
        np.round(np.linspace(380, 3_000_000, num=7)).astype(int),
    )
)
print(sorted(sample_sizes))

for sample_size in sorted(sample_sizes):
    print(f"Sample Size: {sample_size}", flush=True)
    dgp_factory_list = [factory(N=sample_size) for factory in dgp_factory_class_list]
    for dgp_factory in dgp_factory_list:
        dgp_name = type(dgp_factory.create_dgp()).__name__
        print(f"\nDGP name: {dgp_name}", flush=True)
        for it in tqdm(range(NUM_ITERS)):
            result = plan.execute(dgp_factory, seed=it * 1001)
            result["iteration"] = it
            result["sample_size"] = sample_size
            result["dgp"] = dgp_name
            filename = f"time-results/{dgp_name}_n{sample_size}_i{it}.csv.gz"
            result.to_csv(filename, index=False)
            upload_file(filename, BUCKET)
            dfs.append(result)